# 06: Appendix: barley seeds as voxel complexes

Paper section: Appendix, *Barley Seeds as Voxel Complexes*

Section 4 computes the DIWECT of each seed's surface mesh, whereas Amézquita et al. (2022) compute the ECT of the cubical complex of the voxels. Here the DIWECT is computed on the Freudenthal triangulation of each seed's cubical complex

In [1]:
import os
import time

import numpy as np
import torch
from joblib import Parallel, delayed

import barley as B
from diwect import DIWECT

RECOMPUTE = False
VERIFY = 2
N_JOBS = max(1, (os.cpu_count() or 2) - 2)
NUM_HEIGHTS_VOXEL_DIWECT = 32
CACHE = B.CACHE / "voxel_diwect.npz"

d = B.load_table(with_files=True)
files, accession, panicle = d["files"], d["accession"], d["panicle"]
check = np.random.default_rng(B.SEED).choice(len(files), 20, replace=False)

## The triangulation has the cubical complex's ECFs

In [2]:
img = B.load_seed(files[check[0]])
cv, cc = B.cubical_complex(img)
tv, tc = B.freudenthal(img)
c0 = cv.mean(0)
cv, tv = cv - c0, tv - c0
ec = lambda h, cells, t: (h <= t).sum() + sum((-1) ** k * (h[c].max(1) <= t).sum() for k, c in enumerate(cells, 1))
rng = np.random.default_rng(B.SEED)
test_dirs = [v / np.linalg.norm(v) for v in rng.normal(size=(30, 3))] + list(np.eye(3)) + [np.ones(3) / np.sqrt(3)]
heights = np.linspace(-45, 45, 61)
mismatch = sum(ec(cv @ v, cc, t) != ec(tv @ v, tc, t) for v in test_dirs for t in heights)
print(f"voxel complex: {len(cv)} vertices, cells {[len(c) for c in cc]}; Freudenthal triangulation: "
      f"simplices {[len(c) for c in tc]}")
print(f"Euler characteristic curves differ at {mismatch} of {len(test_dirs) * len(heights)} (direction, height) pairs")
assert mismatch == 0

voxel complex: 53186 vertices, cells [153726, 148051, 47510]; Freudenthal triangulation: simplices [349287, 581162, 285060]
Euler characteristic curves differ at 0 of 2074 (direction, height) pairs


## Voxel DIWECT of every seed

In [3]:
def process_voxel_diwect(i, path):
    torch.set_num_threads(1)
    verts, cells = B.freudenthal(B.load_seed(path))
    verts = verts - verts.mean(0)
    one = lambda n: torch.ones(n, dtype=torch.float64)
    cx = [(torch.tensor(verts), one(len(verts)))] + [(torch.tensor(c), one(len(c))) for c in cells]
    with torch.no_grad():
        return DIWECT(NUM_HEIGHTS_VOXEL_DIWECT, normalize=True)(cx).numpy()


if RECOMPUTE or not CACHE.exists():
    t0 = time.time()
    voxel_diwect = np.stack(Parallel(n_jobs=N_JOBS)(delayed(process_voxel_diwect)(i, p) for i, p in enumerate(files)))
    np.savez_compressed(CACHE, diwect=voxel_diwect)
    print(f"voxel DIWECT: {time.time() - t0:.0f}s")
voxel_diwect = np.load(CACHE)["diwect"]

if VERIFY:
    sample = check[:VERIFY]
    new = np.stack(Parallel(n_jobs=N_JOBS)(delayed(process_voxel_diwect)(i, files[i]) for i in sample))
    print(f"recomputed {VERIFY} seeds: max abs difference from the cache {np.abs(new - voxel_diwect[sample]).max():.1e}")
    assert np.allclose(new, voxel_diwect[sample], rtol=1e-6, atol=1e-6)

recomputed 2 seeds: max abs difference from the cache 1.1e-06


## Classification: voxel DIWECT vs mesh DIWECT

In [4]:
splits = B.make_splits(accession, panicle)
mesh_diwect = np.load(B.CACHE / "features.npz")["diwect"]
for name, x in [("DIWECT, voxel complex (32 heights)", voxel_diwect), ("DIWECT, surface mesh (64 heights)", mesh_diwect)]:
    r = B.evaluate(B.block(x), accession, splits, n_jobs=N_JOBS)
    print(f"{name:>36}: " + "   ".join(f"{s} split {r[s].mean():.3f} ± {r[s].std():.3f}" for s in r))

  DIWECT, voxel complex (32 heights): seed split 0.395 ± 0.015   spike split 0.222 ± 0.020


   DIWECT, surface mesh (64 heights): seed split 0.390 ± 0.012   spike split 0.238 ± 0.020
